In [4]:
import time
import json
import pandas as pd
import requests
from datetime import datetime
import undetected_chromedriver as uc

# ---------------------------------------------------
# USER INPUT REQUIRED — fill this after you send me list
# ---------------------------------------------------
CITY_URLS = {
    # "Delhi": [""https://aqicn.org/city/delhi/"]
}
OUTPUT_FILE = "aqi_2025_history.xlsx"

# ---------------------------------------------------
# Extract AQICN API JSON endpoint (hidden)
# ---------------------------------------------------
def get_api_endpoint(city_url, wait_sec=7):
    driver = uc.Chrome()
    driver.get(city_url)
    time.sleep(wait_sec)

    logs = driver.get_log("performance")
    driver.quit()

    for entry in logs:
        try:
            msg = json.loads(entry["message"])
            method = msg["message"]["method"]
            if method == "Network.responseReceived":
                url = msg["message"]["params"]["response"]["url"]
                if "feed" in url and "obs" in url:
                    return url
        except:
            pass
    return None


# ---------------------------------------------------
# Fetch AQI historical data (AQICN JSON)
# ---------------------------------------------------
def fetch_history(api_url):
    try:
        r = requests.get(api_url, timeout=20).json()
    except:
        return None

    if "data" not in r:
        return None

    # daily historical PM2.5
    daily = r["data"]["forecast"]["daily"].get("pm25", [])

    rows = []
    for day in daily:
        rows.append({
            "date": day["day"],
            "avg": day["avg"],
            "min": day["min"],
            "max": day["max"],
            "source_url": api_url
        })

    return pd.DataFrame(rows)


# ---------------------------------------------------
# Main Workflow — 100 cities
# ---------------------------------------------------
all_frames = []

for state, urls in CITY_URLS.items():
    print(f"\n========== STATE: {state} ==========")
    for city_url in urls:
        print(f"\n→ Processing: {city_url}")

        # STEP 1 — extract API endpoint
        api = get_api_endpoint(city_url)

        if not api:
            print("  ❌ API endpoint not found, retrying once...")
            api = get_api_endpoint(city_url, wait_sec=10)

        if not api:
            print("  ❌ FAILED: skipping this city.\n")
            continue

        print("  ✓ API:", api)

        # STEP 2 — fetch historical AQI data
        df = fetch_history(api)
        if df is None or df.empty:
            print("  ❌ No historical data found.")
            continue

        df["state"] = state
        df["city_page"] = city_url

        all_frames.append(df)

# ---------------------------------------------------
# Combine all states + cities
# ---------------------------------------------------
if all_frames:
    final_df = pd.concat(all_frames, ignore_index=True)

    # convert date column
    final_df["date"] = pd.to_datetime(final_df["date"], errors="coerce")

    # keep only year 2025
    final_df = final_df[final_df["date"].dt.year == 2025]

    # Append to Excel (or create)
    try:
        old = pd.read_excel(OUTPUT_FILE)
        combo = pd.concat([old, final_df], ignore_index=True)
        combo.drop_duplicates(subset=["date", "city_page"], keep="last", inplace=True)
    except FileNotFoundError:
        combo = final_df

    combo.to_excel(OUTPUT_FILE, index=False)
    print("\n✅ Saved updated file:", OUTPUT_FILE)
else:
    print("\n❌ No data collected.")



❌ No data collected.


In [1]:
import requests
import pandas as pd
import os
import time
from datetime import datetime

# -------------------------
# CITY LISTS YOU PROVIDED
# -------------------------
STATE_CITY_MAP = {
    "Delhi": ["Delhi", "New Delhi", "Anand Vihar", "Ashok Vihar"],
    "Maharashtra": ["Mumbai", "Pune", "Nagpur", "Nashik"],
    "Uttar_Pradesh": ["Lucknow", "Ghaziabad", "Noida", "Kanpur"],
    "Haryana": ["Gurugram", "Faridabad", "Hisar", "Panipat"],
    "Karnataka": ["Bengaluru", "Mysuru", "Mangalore", "Hubli"],
    "West_Bengal": ["Kolkata", "Howrah", "Asansol", "Durgapur"],
    "Gujarat": ["Ahmedabad", "Surat", "Vadodara", "Rajkot"],
    "Punjab": ["Ludhiana", "Amritsar", "Jalandhar", "Patiala"],
    "Kerala": ["Kochi", "Thiruvananthapuram", "Kozhikode", "Thrissur"],
    "Rajasthan": ["Jaipur", "Udaipur", "Kota", "Jodhpur"],
}

OUTPUT_FILE = "aqi_in_2025.xlsx"


# -------------------------
# Convert city → API slug
# -------------------------
def make_slug(city):
    return city.lower().replace(" ", "-").replace("_", "-")


# -------------------------
# Fetch historical AQI for a city
# -------------------------
def fetch_city_data(city_name):
    slug = make_slug(city_name)

    url = (
        f"https://api.aqi.in/api/v1/cities/{slug}/daily-aqi"
        "?from=2025-01-01&to=" + datetime.now().strftime("%Y-%m-%d")
    )

    for attempt in range(3):
        try:
            r = requests.get(url, timeout=20)
            if r.status_code == 200:
                return r.json()
        except:
            time.sleep(2)

    return None


# -------------------------
# Append rows directly to Excel
# -------------------------
def append_to_excel(df):
    if not os.path.exists(OUTPUT_FILE):
        df.to_excel(OUTPUT_FILE, index=False)
    else:
        old = pd.read_excel(OUTPUT_FILE)
        final = pd.concat([old, df], ignore_index=True)
        final.drop_duplicates(subset=["date", "city"], keep="last", inplace=True)
        final.to_excel(OUTPUT_FILE, index=False)


# -------------------------
# MAIN EXECUTION
# -------------------------
print("\n🚀 Starting AQI.in historical scraper...\n")

for state, cities in STATE_CITY_MAP.items():
    for city in cities:
        print(f"\n📍 Fetching: {state} → {city}")

        data = fetch_city_data(city)
        if not data or "data" not in data:
            print(f"   ❌ Failed: {city}")
            continue

        daily = data["data"]

        if len(daily) == 0:
            print(f"   ⚠ No daily data found")
            continue

        rows = []
        for item in daily:
            rows.append({
                "state": state,
                "city": city,
                "date": item.get("date"),
                "aqi": item.get("aqi"),
                "pm25": item.get("pm25"),
                "pm10": item.get("pm10"),
                "dominant_pollutant": item.get("dominantPollutant"),
                "category": item.get("aqiCategory")
            })

        df = pd.DataFrame(rows)
        append_to_excel(df)

        print(f"   ✓ Saved {len(rows)} daily entries")

print("\n✅ DONE — Saved complete historical AQI for 2025 so far.")



🚀 Starting AQI.in historical scraper...


📍 Fetching: Delhi → Delhi
   ❌ Failed: Delhi

📍 Fetching: Delhi → New Delhi
   ❌ Failed: New Delhi

📍 Fetching: Delhi → Anand Vihar
   ❌ Failed: Anand Vihar

📍 Fetching: Delhi → Ashok Vihar
   ❌ Failed: Ashok Vihar

📍 Fetching: Maharashtra → Mumbai
   ❌ Failed: Mumbai

📍 Fetching: Maharashtra → Pune
   ❌ Failed: Pune

📍 Fetching: Maharashtra → Nagpur
   ❌ Failed: Nagpur

📍 Fetching: Maharashtra → Nashik
   ❌ Failed: Nashik

📍 Fetching: Uttar_Pradesh → Lucknow
   ❌ Failed: Lucknow

📍 Fetching: Uttar_Pradesh → Ghaziabad
   ❌ Failed: Ghaziabad

📍 Fetching: Uttar_Pradesh → Noida
   ❌ Failed: Noida

📍 Fetching: Uttar_Pradesh → Kanpur
   ❌ Failed: Kanpur

📍 Fetching: Haryana → Gurugram
   ❌ Failed: Gurugram

📍 Fetching: Haryana → Faridabad
   ❌ Failed: Faridabad

📍 Fetching: Haryana → Hisar
   ❌ Failed: Hisar

📍 Fetching: Haryana → Panipat
   ❌ Failed: Panipat

📍 Fetching: Karnataka → Bengaluru
   ❌ Failed: Bengaluru

📍 Fetching: Karnataka → Mysuru


In [9]:
import requests
import pandas as pd
from datetime import datetime
import time

TOKEN = "58fdf2e5db777f93f7d481b664ab91b06ebd35eb"

STATE_CITY_MAP = {
    "Delhi": ["Delhi", "New Delhi", "Anand Vihar", "Ashok Vihar"],
    "Maharashtra": ["Mumbai", "Pune", "Nagpur", "Nashik"],
    "Uttar_Pradesh": ["Lucknow", "Ghaziabad", "Noida", "Kanpur"],
    "Haryana": ["Gurugram", "Faridabad", "Hisar", "Panipat"],
    "Karnataka": ["Bengaluru", "Mysuru", "Mangalore", "Hubli"],
    "West_Bengal": ["Kolkata", "Howrah", "Asansol", "Durgapur"],
    "Gujarat": ["Ahmedabad", "Surat", "Vadodara", "Rajkot"],
    "Punjab": ["Ludhiana", "Amritsar", "Jalandhar", "Patiala"],
    "Kerala": ["Kochi", "Thiruvananthapuram", "Kozhikode", "Thrissur"],
    "Rajasthan": ["Jaipur", "Udaipur", "Kota", "Jodhpur"],
}

def get_station_id(city):
    """Return nearest monitoring station ID for a city."""
    url = f"https://api.waqi.info/feed/{city}/?token={TOKEN}"
    r = requests.get(url).json()
    if r["status"] != "ok":
        return None
    return r["data"]["idx"]

def get_station_history(idx):
    """Fetch full historical observations (hourly) for a station."""
    url = f"https://api.waqi.info/api/feed/@{idx}/obs.en.json?token={TOKEN}"
    r = requests.get(url).json()

    if "rxs" not in r:
        return None

    rows = []
    for entry in r["rxs"]:
        if "time" not in entry or "iaqi" not in entry.get("msg", {}):
            continue

        timestamp = entry["time"]["iso"]
        iaqi = entry["msg"]["iaqi"]

        rows.append({
            "timestamp": timestamp,
            "aqi": iaqi.get("pm25", {}).get("v"),
            "pm25": iaqi.get("pm25", {}).get("v"),
            "pm10": iaqi.get("pm10", {}).get("v"),
            "no2": iaqi.get("no2", {}).get("v"),
            "o3": iaqi.get("o3", {}).get("v"),
            "so2": iaqi.get("so2", {}).get("v"),
            "co": iaqi.get("co", {}).get("v"),
        })

    df = pd.DataFrame(rows)
    if df.empty:
        return None

    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df["date"] = df["timestamp"].dt.date
    return df


def main():
    final_rows = []

    for state, cities in STATE_CITY_MAP.items():
        for city in cities:
            print(f"📍 {state} → {city}")

            idx = get_station_id(city)
            if idx is None:
                print("   ❌ No station found")
                continue

            print(f"   ✓ Station ID: {idx}")

            df = get_station_history(idx)
            if df is None:
                print("   ❌ No historical data")
                continue

            df["state"] = state
            df["city"] = city

            # filter only 2025
            df_2025 = df[df["timestamp"].dt.year == 2025].copy()

            if df_2025.empty:
                print("   ⚠ No 2025 data")
                continue

            final_rows.append(df_2025)
            print(f"   ✓ Retrieved {len(df_2025)} rows")

            time.sleep(2)

    if final_rows:
        out = pd.concat(final_rows, ignore_index=True)
        out.to_excel("aqi_waqi_2025.xlsx", index=False)
        print("\n🎉 Saved:", len(out), "rows to aqi_waqi_2025.xlsx")
    else:
        print("No data fetched.")


if __name__ == "__main__":
    main()


📍 Delhi → Delhi
   ✓ Station ID: 10111
   ❌ No historical data
📍 Delhi → New Delhi
   ✓ Station ID: 10122
   ❌ No historical data
📍 Delhi → Anand Vihar
   ❌ No station found
📍 Delhi → Ashok Vihar
   ❌ No station found
📍 Maharashtra → Mumbai
   ✓ Station ID: 12454
   ❌ No historical data
📍 Maharashtra → Pune
   ✓ Station ID: 3760
   ❌ No historical data
📍 Maharashtra → Nagpur
   ✓ Station ID: 12444
   ❌ No historical data
📍 Maharashtra → Nashik
   ✓ Station ID: 11283
   ❌ No historical data
📍 Uttar_Pradesh → Lucknow
   ✓ Station ID: 3845
   ❌ No historical data
📍 Uttar_Pradesh → Ghaziabad
   ✓ Station ID: 11899
   ❌ No historical data
📍 Uttar_Pradesh → Noida
   ✓ Station ID: 11863
   ❌ No historical data
📍 Uttar_Pradesh → Kanpur
   ✓ Station ID: 13727
   ❌ No historical data
📍 Haryana → Gurugram
   ❌ No station found
📍 Haryana → Faridabad
   ✓ Station ID: 12813
   ❌ No historical data
📍 Haryana → Hisar
   ✓ Station ID: 11855
   ❌ No historical data
📍 Haryana → Panipat
   ❌ No station fo

In [11]:
import requests
import pandas as pd
from datetime import datetime
import time

TOKEN = "58fdf2e5db777f93f7d481b664ab91b06ebd35eb"

CITIES = [
    "Delhi",
    "Anand Vihar",
    "Mumbai",
    "Pune",
    "Kolkata",
    "Chennai",
]

OUTPUT_FILE = "waqi_historical_india.xlsx"

# ---------------------------------------------
# STEP 1 → Find Station ID from City Name
# ---------------------------------------------
def find_station_id(city):
    url = f"https://api.waqi.info/search/?token={TOKEN}&keyword={city}"
    r = requests.get(url).json()

    if r.get("status") != "ok":
        print(f"❌ Search failed for {city}")
        return None

    results = r.get("data", [])
    if not results:
        print(f"❌ No station found for {city}")
        return None

    # Pick best station with max relevance
    best = results[0]
    station_id = best["uid"]
    station_name = best["station"]["name"]

    print(f"   ✓ Found station: {station_name} (ID: {station_id})")
    return station_id


# ---------------------------------------------
# STEP 2 → Fetch Historical Hourly Data
# ---------------------------------------------
def fetch_station_history(station_id):
    url = f"https://api.waqi.info/api/feed/@{station_id}/obs.en.json?token={TOKEN}"
    r = requests.get(url).json()

    if r.get("status") != "ok":
        print(f"   ❌ Failed fetching history for {station_id}")
        return None

    history = r["rxs"]["obs"]  # list of hourly readings
    rows = []

    for entry in history:
        t = entry.get("t")  # timestamp
        iaqi = entry.get("iaqi", {})

        rows.append({
            "datetime": t,
            "aqi": entry.get("aqi"),

            "pm25": iaqi.get("pm25", {}).get("v"),
            "pm10": iaqi.get("pm10", {}).get("v"),
            "no2": iaqi.get("no2", {}).get("v"),
            "o3": iaqi.get("o3", {}).get("v"),
            "so2": iaqi.get("so2", {}).get("v"),
            "co": iaqi.get("co", {}).get("v"),
        })

    df = pd.DataFrame(rows)
    df["datetime"] = pd.to_datetime(df["datetime"], errors="ignore")
    df["date"] = df["datetime"].dt.date

    print(f"   ✓ Retrieved {len(df)} hourly records")
    return df


# ---------------------------------------------
# STEP 3 → Resample to daily average
# ---------------------------------------------
def to_daily(df):
    return df.groupby("date").mean(numeric_only=True).reset_index()


# ---------------------------------------------
# MAIN PIPELINE
# ---------------------------------------------
final = []

for city in CITIES:
    print(f"\n📍 Fetching city: {city}")

    station_id = find_station_id(city)
    if not station_id:
        continue

    df_hourly = fetch_station_history(station_id)
    if df_hourly is None or df_hourly.empty:
        continue

    df_hourly["city"] = city
    df_daily = to_daily(df_hourly)
    df_daily["city"] = city

    final.append(df_daily)

    time.sleep(1)  # WAQI rate limiting protection

# Combine all cities
if final:
    full = pd.concat(final, ignore_index=True)
    full.to_excel(OUTPUT_FILE, index=False)
    print(f"\n✅ DONE — Saved {len(full)} daily rows to {OUTPUT_FILE}")
else:
    print("\n❌ No data collected")



📍 Fetching city: Delhi
   ✓ Found station: Delhi Institute of Tool Engineering, Wazirpur, Delhi, Delhi, India (ID: 10114)
   ❌ Failed fetching history for 10114

📍 Fetching city: Anand Vihar
   ✓ Found station: Anand Vihar, Hapur, India (ID: 11318)
   ❌ Failed fetching history for 11318

📍 Fetching city: Mumbai
   ✓ Found station: Deonar, Mumbai, India (ID: 13712)
   ❌ Failed fetching history for 13712

📍 Fetching city: Pune
   ✓ Found station: Shivajinagar, Pune, Pune, India (ID: 3760)
   ❌ Failed fetching history for 3760

📍 Fetching city: Kolkata
   ✓ Found station: Padmapukur, Howrah, India (ID: 11320)
   ❌ Failed fetching history for 11320

📍 Fetching city: Chennai
   ✓ Found station: Manali Village, Chennai, Chennai, India (ID: 11859)
   ❌ Failed fetching history for 11859

❌ No data collected


In [38]:
import requests

key = "bdb211c584ad8c0f75b3afc12181cbc3de3c2456d5ad1699937eb836ef123985"
sensor_id = 3917

url = f"https://api.openaq.org/v3/sensors/{sensor_id}/days"

headers = {
    "X-API-Key": key
}

params = {
    "date_from": "2025-01-01T00:00:00Z",
    "date_to": "2025-11-19T23:59:59Z"
}

r = requests.get(url, headers=headers, params=params).json()
print(r)


{'meta': {'name': 'openaq-api', 'website': '/', 'page': 1, 'limit': 100, 'found': '>100'}, 'results': [{'value': 0.0262, 'flagInfo': {'hasFlags': False}, 'parameter': {'id': 10, 'name': 'o3', 'units': 'ppm', 'displayName': None}, 'period': {'label': '1day', 'interval': '24:00:00', 'datetimeFrom': {'utc': '2025-01-01T07:00:00Z', 'local': '2025-01-01T00:00:00-07:00'}, 'datetimeTo': {'utc': '2025-01-02T07:00:00Z', 'local': '2025-01-02T00:00:00-07:00'}}, 'coordinates': None, 'summary': {'min': 0.004, 'q02': 0.004, 'q25': 0.010499999999999999, 'median': 0.0285, 'q75': 0.03775, 'q98': 0.047, 'max': 0.047, 'avg': 0.026208333333333337, 'sd': 0.00960339912350178}, 'coverage': {'expectedCount': 24, 'expectedInterval': '24:00:00', 'observedCount': 24, 'observedInterval': '24:00:00', 'percentComplete': 100.0, 'percentCoverage': 100.0, 'datetimeFrom': {'utc': '2025-01-01T08:00:00Z', 'local': '2025-01-01T01:00:00-07:00'}, 'datetimeTo': {'utc': '2025-01-02T07:00:00Z', 'local': '2025-01-02T00:00:00-07

In [24]:
import requests

key = "bdb211c584ad8c0f75b3afc12181cbc3de3c2456d5ad1699937eb8363ef123985"

STATE_CITY_MAP = {
    "Delhi": ["Delhi", "New Delhi", "Anand Vihar", "Ashok Vihar"],
    "Maharashtra": ["Mumbai", "Pune", "Nagpur", "Nashik"],
    "Uttar_Pradesh": ["Lucknow", "Ghaziabad", "Noida", "Kanpur"],
    "Haryana": ["Gurugram", "Faridabad", "Hisar", "Panipat"],
    "Karnataka": ["Bengaluru", "Mysuru", "Mangalore", "Hubli"],
    "West_Bengal": ["Kolkata", "Howrah", "Asansol", "Durgapur"],
    "Gujarat": ["Ahmedabad", "Surat", "Vadodara", "Rajkot"],
    "Punjab": ["Ludhiana", "Amritsar", "Jalandhar", "Patiala"],
    "Kerala": ["Kochi", "Thiruvananthapuram", "Kozhikode", "Thrissur"],
    "Rajasthan": ["Jaipur", "Udaipur", "Kota", "Jodhpur"],
}

# Step 1: fetch all sensors for India
all_sensors = []
page = 1
while True:
    url = "https://api.openaq.org/v3/sensors"
    headers = {"X-API-Key": key}
    params = {
        "country": "IN",
        "limit": 100,
        "page": page
    }

    response = requests.get(url, headers=headers, params=params)
    data = response.json()

    if "results" not in data or len(data["results"]) == 0:
        break

    all_sensors.extend(data["results"])
    page += 1

print(f"Total sensors fetched: {len(all_sensors)}")

# Step 2: filter sensors by STATE_CITY_MAP
filtered_sensors = []
for state, cities in STATE_CITY_MAP.items():
    for sensor in all_sensors:
        if sensor.get("city") in cities:
            filtered_sensors.append({
                "state": state,
                "city": sensor.get("city"),
                "sensor_id": sensor["id"],
                "sensor_name": sensor.get("name", "")
            })

# Print filtered sensors
for s in filtered_sensors:
    print(s)


Total sensors fetched: 0


In [22]:
for sensor in all_sensors:
    sensor_id = sensor["sensor_id"]
    url = f"https://api.openaq.org/v3/sensors/{sensor_id}/days"
    params = {
        "date_from": "2025-01-01T00:00:00Z",
        "date_to": "2025-11-19T23:59:59Z"
    }
    response = requests.get(url, headers=headers, params=params)
    data = response.json()
    # Save or process data


In [23]:
data

{'detail': 'Not Found'}

In [80]:
key = "19639b8ce99d6550f1bae601c1f446074e08f9ab17305fd212354dbce0817522"

In [85]:
url = "https://api.openaq.org/v3/locations"

headers = {"X-API-Key": key}

params = {
    "country_id": 9,  # India
    "limit": 100,     # max 100 per page
    "page": 1
}

r = requests.get(url, headers=headers, params=params).json()
r

{'meta': {'name': 'openaq-api',
  'website': '/',
  'page': 1,
  'limit': 100,
  'found': '>100'},
 'results': [{'id': 3,
   'name': 'NMA - Nima',
   'locality': None,
   'timezone': 'Africa/Accra',
   'country': {'id': 152, 'code': 'GH', 'name': 'Ghana'},
   'owner': {'id': 4, 'name': 'Unknown Governmental Organization'},
   'provider': {'id': 209, 'name': 'Dr. Raphael E. Arku and Colleagues'},
   'isMobile': False,
   'isMonitor': True,
   'instruments': [{'id': 2, 'name': 'Government Monitor'}],
   'sensors': [{'id': 6,
     'name': 'pm10 µg/m³',
     'parameter': {'id': 1,
      'name': 'pm10',
      'units': 'µg/m³',
      'displayName': 'PM10'}},
    {'id': 5,
     'name': 'pm25 µg/m³',
     'parameter': {'id': 2,
      'name': 'pm25',
      'units': 'µg/m³',
      'displayName': 'PM2.5'}}],
   'coordinates': {'latitude': 5.58389, 'longitude': -0.19968},
   'licenses': None,
   'bounds': [-0.19968, 5.58389, -0.19968, 5.58389],
   'distance': None,
   'datetimeFirst': None,
   'da

In [86]:
url = "https://api.openaq.org/v3/countries"
r = requests.get(url, headers=headers).json()
print(r)


{'meta': {'name': 'openaq-api', 'website': '/', 'page': 1, 'limit': 100, 'found': 142}, 'results': [{'id': 1, 'code': 'ID', 'name': 'Indonesia', 'datetimeFirst': '2016-01-30T01:00:00Z', 'datetimeLast': '2025-11-20T18:00:00Z', 'parameters': [{'id': 1, 'name': 'pm10', 'units': 'µg/m³', 'displayName': None}, {'id': 2, 'name': 'pm25', 'units': 'µg/m³', 'displayName': None}, {'id': 3, 'name': 'o3', 'units': 'µg/m³', 'displayName': None}, {'id': 10, 'name': 'o3', 'units': 'ppm', 'displayName': None}, {'id': 11, 'name': 'bc', 'units': 'µg/m³', 'displayName': None}, {'id': 15, 'name': 'no2', 'units': 'ppb', 'displayName': None}, {'id': 19, 'name': 'pm1', 'units': 'µg/m³', 'displayName': None}, {'id': 21, 'name': 'co2', 'units': 'ppm', 'displayName': None}, {'id': 24, 'name': 'no', 'units': 'ppb', 'displayName': None}, {'id': 98, 'name': 'relativehumidity', 'units': '%', 'displayName': None}, {'id': 100, 'name': 'temperature', 'units': 'c', 'displayName': None}, {'id': 125, 'name': 'um003', 'un

In [101]:
for row in r['results']:
    for item in row.items():
        if 'India' in item:
            print(row)

{'id': 9, 'code': 'IN', 'name': 'India', 'datetimeFirst': '2016-01-30T00:30:00Z', 'datetimeLast': '2025-11-20T18:00:00Z', 'parameters': [{'id': 1, 'name': 'pm10', 'units': 'µg/m³', 'displayName': None}, {'id': 2, 'name': 'pm25', 'units': 'µg/m³', 'displayName': None}, {'id': 3, 'name': 'o3', 'units': 'µg/m³', 'displayName': None}, {'id': 4, 'name': 'co', 'units': 'µg/m³', 'displayName': None}, {'id': 5, 'name': 'no2', 'units': 'µg/m³', 'displayName': None}, {'id': 6, 'name': 'so2', 'units': 'µg/m³', 'displayName': None}, {'id': 10, 'name': 'o3', 'units': 'ppm', 'displayName': None}, {'id': 15, 'name': 'no2', 'units': 'ppb', 'displayName': None}, {'id': 19, 'name': 'pm1', 'units': 'µg/m³', 'displayName': None}, {'id': 22, 'name': 'wind_direction', 'units': 'deg', 'displayName': None}, {'id': 23, 'name': 'nox', 'units': 'ppb', 'displayName': None}, {'id': 24, 'name': 'no', 'units': 'ppb', 'displayName': None}, {'id': 34, 'name': 'wind_speed', 'units': 'm/s', 'displayName': None}, {'id': 

In [ ]:
url = "https://api.openaq.org/v3/locations"
params = {
    "country": 9,  # India
    "limit": 100,  # page size
    "page": 1
}

r = requests.get(url, headers={"X-API-Key": key}).json()
locations = r.get("results", [])


In [118]:
locations

[]

In [110]:
all_sensors = []

for loc in locations:
    city = loc.get("city")
    state = loc.get("region")  # sometimes available
    for sensor in loc.get("sensors", []):
        all_sensors.append({
            "state": state,
            "city": city,
            "location_id": loc["id"],
            "location_name": loc.get("name"),
            "sensor_id": sensor["id"],
            "parameter": sensor.get("parameter"),
        })

print(all_sensors[:5])


[{'state': None, 'city': None, 'location_id': 3, 'location_name': 'NMA - Nima', 'sensor_id': 6, 'parameter': {'id': 1, 'name': 'pm10', 'units': 'µg/m³', 'displayName': 'PM10'}}, {'state': None, 'city': None, 'location_id': 3, 'location_name': 'NMA - Nima', 'sensor_id': 5, 'parameter': {'id': 2, 'name': 'pm25', 'units': 'µg/m³', 'displayName': 'PM2.5'}}, {'state': None, 'city': None, 'location_id': 4, 'location_name': 'NMT - Nima', 'sensor_id': 7, 'parameter': {'id': 1, 'name': 'pm10', 'units': 'µg/m³', 'displayName': 'PM10'}}, {'state': None, 'city': None, 'location_id': 4, 'location_name': 'NMT - Nima', 'sensor_id': 8, 'parameter': {'id': 2, 'name': 'pm25', 'units': 'µg/m³', 'displayName': 'PM2.5'}}, {'state': None, 'city': None, 'location_id': 5, 'location_name': 'JTA - Jamestown', 'sensor_id': 10, 'parameter': {'id': 1, 'name': 'pm10', 'units': 'µg/m³', 'displayName': 'PM10'}}]
